# Entity fingerprints

An entity can appear on many rows. An account might place orders
through several channels, or a device might send events of different
kinds. Name `entity_fingerprints` to compare the **mix within each
entity**. The results can describe groups, a continuum, entities
whose mix differs from their group, specialists, or a change in
mix along a resolved axis.

Supply `entity` and mark categorical columns with
`roles={column: "dimension"}` when inference needs help.
A time column or an `ordered_axis` enables drift. Use
`onceover.infer_spec()` to inspect or edit those choices before
profiling.

## Count activity or weight it

These accounts place the same number of orders on each plan.
Twenty spend more per order on Basic, and twenty spend more per
order on Plus. Basic accounts are three times larger by spend,
so the pooled spend shares need not match the shares of accounts.

In [1]:
import polars as pl
import onceover

orders = pl.DataFrame(
    [
        (account, plan, (3 if account < 20 else 1) * (9 if plan == favorite else 1))
        for account in range(40)
        for favorite in ["Basic" if account < 20 else "Plus"]
        for plan in ["Basic", "Plus"]
        for _ in range(60)
    ],
    schema=["account", "plan", "spend"],
    orient="row",
)
hints = {"entity": "account", "roles": {"plan": "dimension", "spend": "measure"}}
counts = onceover.profile(orders, hints=hints, detectors=["entity_fingerprints"])
print(counts.coverage["entity_fingerprints"]["detail"])

entity account; weight: row count; drift skipped: no resolved axis with per-row keys; dimensions: plan: entities do not differ in mix beyond sampling noise (excess V 0.000, chi-square 0 on 39 df); dimension scans: 1 of 1 started (0 pair candidates before caps)


In [2]:
spec = onceover.infer_spec(orders, hints={**hints, "weights": ["spend"]})
weighted = onceover.profile(orders, spec=spec, detectors=["entity_fingerprints"])
for finding in weighted.select(type="entity_groups"):
    print(finding["claim"])

account fall into 2 groups by their mix of plan weighted by 'spend': 20 account (50%) with 90% on 'Plus' (against 30% overall); 20 account (50%) with 90% on 'Basic' (against 70% overall)


Each group's size counts accounts. The percentages inside the
description pool the spend of those accounts. The overall
percentages pool spend across the examined accounts. The two
groups each have 20 accounts, but the overall spend is 70% Basic
and 30% Plus.

A weight column specifies a separate view of the mix. If several
weights apply, each is analyzed separately. To compare row shares
with weight shares, make one call without weights and another
with weights. Coverage records any refused weights. Missing and
zero weights contribute no mass. To keep a single large order
from looking like ample evidence, the model discounts support by
the weight imbalance. An entity's effective row count is
`sum(weight)**2 / sum(weight**2)`. Weighted and joint views
require at least ten effective rows per entity.

## Compare pairings, then look for changes

In the next example, every account has equal shares by plan
and by channel. Half the accounts order Basic online and Plus
in store, and half do the reverse. The pair carries information
that neither dimension carries alone.

In [3]:
def paired_orders(switching):
    rows = []
    for account in range(40):
        for period in (1, 2):
            aligned = (account < 20) != (switching and period == 2)
            for plan, channel in (
                ("Basic", "Web"), ("Basic", "Store"),
                ("Plus", "Web"), ("Plus", "Store"),
            ):
                if ((plan == "Basic") == (channel == "Web")) == aligned:
                    rows.extend([(account, period, plan, channel)] * 100)
    return pl.DataFrame(rows, schema=["account", "period", "plan", "channel"], orient="row")


pair_hints = {
    "entity": "account",
    "ordered_axis": "period",
    "roles": {"plan": "dimension", "channel": "dimension"},
}
static = onceover.profile(
    paired_orders(False), hints=pair_hints, detectors=["entity_fingerprints"],
)
for finding in static.select(type="entity_groups"):
    print(finding["claim"])

account fall into 2 groups by their mix of channel × plan: 20 account (50%) with 50% on '"Web" × "Basic"' (against 25% overall), 50% on '"Store" × "Plus"' (against 25% overall); 20 account (50%) with 50% on '"Store" × "Basic"' (against 25% overall), 50% on '"Web" × "Plus"' (against 25% overall)


If both groups switch pairings in the second period, the annual
mix of every account becomes the same. Temporal evidence can
still find the changes. The peer baseline uses accounts observed
with enough support on both sides of a boundary. A broad shared
shift is reported separately, and windows overlapping that shift
are withheld from individual comparisons.

In [4]:
temporal = onceover.profile(
    paired_orders(True), hints=pair_hints, detectors=["entity_fingerprints"],
)
for finding in temporal.select(type="joint_distribution_change"):
    print(finding["claim"])
    metrics = finding["evidence"][0]["metrics"]
    print(metrics["dimensions"], metrics["n_entity_shifts"])

40 of 40 tested account values changed their channel × plan mix beyond the change among peers. For example, 0 moved from 0% to 50% on '"Store" × "Basic"' around 2 (peers: 26% to 24%)
['channel', 'plan'] 40


## Read coverage before interpreting a null

Coverage names the weights, period starts, and number of entities
tested. The pair screen tries up to 12 pairs in column-name order
and allows up to 30 observed levels per pair. Static pairs must
show additional signal beyond both dimensions. For temporal pairs,
the two component shares must each move by less than 10 percentage
points in the tested windows. Joint changes that also move one of
those shares can therefore be missed.

Temporal comparisons use adjacent windows of up to six periods
on each side. The response depends on axis grain and support.
Treat the statistical results as leads on the entities and windows
the evidence names. Read
[findings and coverage](../usage/coverage.ipynb) for how the
report distinguishes a completed search from a capped one.